# Final Project: Hotel Booking Data Analysis

**Course:** Data Processing with Python (CA25 Customer Projects, HAMK)
**Group members:**
1. Anup Kandel
2. Sandesh Bishokarma
3. Sushant Koirala
4. Tanka Raj Aryal

> **Starter template.** Every block marked `TODO` is work your group must finish.
> The ready-made code covers setup, checks and the required derived variables so you can
> spend your time on the analysis, the decisions and the interpretation, which is where the marks are.
> Before you submit, delete this note and every `TODO` line.

## 1. Context and research questions

**What one row represents:** TODO: one hotel booking (completed or cancelled) for ...

**Population and time period:** TODO: bookings at a city hotel (Lisbon) and a resort hotel (Algarve), Portugal,
with arrival dates from July 2015 to August 2017. This is a 1,000-record sample of the Hotel Booking Demand dataset
(António, de Almeida & Nunes, 2019).

**Purpose of the analysis:** TODO: 2–3 sentences. Who is the audience (hotel management)? What do you want to help them understand?

**Research questions** (together they must include a categorical comparison, a numerical variable,
and a relationship, grouped comparison or time pattern):

1. TODO (categorical comparison), e.g. *Do cancellation rates differ between the City Hotel and the Resort Hotel?*
2. TODO (numerical variable), e.g. *How do lead time and average daily rate differ between cancelled and kept bookings?*
3. TODO (time pattern / relationship), e.g. *How does booking volume vary by arrival month and hotel type?*

## 2. Imports and display settings

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:.2f}".format)

DATA_FILE = "../data/hotel_bookings_final_project.csv"   # the instructor's file: never overwrite it
CLEAN_FILE = "../outputs/hotel_bookings_cleaned.csv"
SUMMARY_FILE = "../outputs/hotel_booking_summary.csv"

## 3. Load and inspect the data

In [ ]:
raw = pd.read_csv(DATA_FILE)   # 'raw' stays unchanged for the whole notebook
print(f"Rows: {raw.shape[0]}, columns: {raw.shape[1]}")
raw.sample(5, random_state=42)

In [ ]:
raw.info()

In [ ]:
# Missing values and exact duplicates
missing = raw.isna().sum()
print("Missing values per column (only columns with any):")
print(missing[missing > 0])
print("\nExact duplicate rows:", raw.duplicated().sum())

In [ ]:
# Useful category counts
for col in ["hotel", "is_canceled", "market_segment", "deposit_type", "customer_type"]:
    print(f"--- {col} ---")
    print(raw[col].value_counts(dropna=False), "\n")

In [ ]:
# Numerical summary statistics
raw.describe().T

## 4. Data-quality assessment and cleaning log

Run the checks below first, then fill in the cleaning log with **your group's decisions**.
There is often no single right answer. You get marks for showing the evidence and justifying the decision.

In [ ]:
# Helper columns used only for the checks
check = raw.copy()
check["nights_tmp"] = check["stays_in_weekend_nights"] + check["stays_in_week_nights"]
check["guests_tmp"] = check["adults"] + check["children"] + check["babies"]
check["date_tmp"] = pd.to_datetime(
    check["arrival_date_year"].astype(str) + "-" + check["arrival_date_month"] + "-"
    + check["arrival_date_day_of_month"].astype(str),
    format="%Y-%B-%d", errors="coerce")

def iqr_upper_fence(series):
    """Return the 1.5*IQR upper fence used to flag unusually large values."""
    q1, q3 = series.quantile([0.25, 0.75])
    return q3 + 1.5 * (q3 - q1)

numeric_cols = raw.select_dtypes("number").columns
checks = {
    "missing values (any column)": raw.isna().any(axis=1).sum(),
    "exact duplicate rows": raw.duplicated().sum(),
    "invalid date combinations": check["date_tmp"].isna().sum(),
    "rows with a negative numeric value": (raw[numeric_cols] < 0).any(axis=1).sum(),
    "zero total guests": (check["guests_tmp"] == 0).sum(),
    "zero total nights": (check["nights_tmp"] == 0).sum(),
    "lead_time above IQR fence": (raw["lead_time"] > iqr_upper_fence(raw["lead_time"])).sum(),
    "average_daily_rate above IQR fence": (raw["average_daily_rate"] > iqr_upper_fence(raw["average_daily_rate"])).sum(),
    "average_daily_rate == 0": (raw["average_daily_rate"] == 0).sum(),
}
pd.Series(checks, name="affected rows").to_frame()

In [ ]:
# Look at the affected rows before deciding. Example: zero-night and zero-guest bookings
check.loc[(check["nights_tmp"] == 0) | (check["guests_tmp"] == 0),
          ["hotel", "is_canceled", "adults", "children", "babies", "nights_tmp", "average_daily_rate", "market_segment"]]

In [ ]:
# TODO: inspect the other issues the same way, e.g.
# raw[raw.duplicated(keep=False)].sort_values(list(raw.columns)).head(10)
# raw.loc[raw["average_daily_rate"] == 0, "market_segment"].value_counts()
# raw.nlargest(5, "lead_time")

### Cleaning log

| Issue checked | Evidence | Decision | Reason |
|---|---|---|---|
| Missing `country` | TODO: n rows | TODO: keep / fill "Unknown" / remove | TODO |
| Exact duplicates | TODO: n rows | TODO | TODO: the dataset has no booking ID, so are these really repeated records, or separate identical bookings (e.g. a group)? |
| Invalid date combinations | TODO | TODO | TODO |
| Negative values | TODO | TODO | TODO |
| Zero total guests | TODO | TODO | TODO |
| Zero total nights | TODO | TODO | TODO |
| Unusual `lead_time` | TODO | TODO | TODO: are long lead times impossible, or just rare? |
| Unusual `average_daily_rate` (incl. 0) | TODO | TODO | TODO: look at `market_segment` (e.g. Complementary) |

In [ ]:
clean = raw.copy()   # cleaned copy, 'raw' stays untouched
rows_before = len(clean)

# TODO: apply YOUR decisions from the cleaning log, for example:
# clean = clean.drop_duplicates()
# clean["country"] = clean["country"].fillna("Unknown")
# clean = clean[(clean["adults"] + clean["children"] + clean["babies"]) > 0]

print(f"Rows before cleaning: {rows_before}, after cleaning: {len(clean)}")

## 5. Transformations and derived variables

In [ ]:
clean["arrival_date"] = pd.to_datetime(
    clean["arrival_date_year"].astype(str) + "-" + clean["arrival_date_month"] + "-"
    + clean["arrival_date_day_of_month"].astype(str),
    format="%Y-%B-%d", errors="coerce")
clean["total_nights"] = clean["stays_in_weekend_nights"] + clean["stays_in_week_nights"]
clean["total_guests"] = clean["adults"] + clean["children"] + clean["babies"]
clean["estimated_booking_value"] = clean["average_daily_rate"] * clean["total_nights"]

# Optional extra variables that may help your questions
clean["cancel_status"] = clean["is_canceled"].map({0: "Not cancelled", 1: "Cancelled"})
clean["arrival_month_num"] = clean["arrival_date"].dt.month

clean[["arrival_date", "total_nights", "total_guests", "average_daily_rate", "estimated_booking_value"]].head()

## 6. Individual-variable analysis

At least **two categorical** variables (counts, %, n, bar chart) and **two numerical** variables
(n, mean, median, std, min, max, histogram or box plot).

In [ ]:
def categorical_summary(df, column):
    """Counts and percentages for one categorical column, with the valid n."""
    counts = df[column].value_counts()
    table = pd.DataFrame({"count": counts, "percent": (counts / counts.sum() * 100).round(1)})
    print(f"{column}: valid n = {counts.sum()}")
    return table

def numerical_summary(df, columns):
    """Valid n, mean, median, std, min and max for numeric columns."""
    return df[columns].agg(["count", "mean", "median", "std", "min", "max"]).T.rename(columns={"count": "valid_n"})

categorical_summary(clean, "hotel")

In [ ]:
# TODO: second categorical variable, e.g. deposit_type or market_segment, with a bar chart

In [ ]:
numerical_summary(clean, ["lead_time", "average_daily_rate"])

In [ ]:
# TODO: histogram or box plot for each numerical variable. Template:
# fig, ax = plt.subplots(figsize=(8, 4))
# ax.hist(clean["lead_time"], bins=30, edgecolor="white")
# ax.set_title(f"Distribution of lead time (n = {clean['lead_time'].notna().sum()})")
# ax.set_xlabel("Lead time (days)"); ax.set_ylabel("Number of bookings")
# plt.show()

## 7. Relationships and grouped analysis

Required: **categorical × categorical** (crosstab, counts + labelled %), **categorical × numerical** (`groupby` + aggregation),
and **numerical × numerical or time-based**.

In [ ]:
# Categorical x categorical: hotel x cancellation
counts = pd.crosstab(clean["hotel"], clean["cancel_status"], margins=True, margins_name="Total")
row_pct = (pd.crosstab(clean["hotel"], clean["cancel_status"], normalize="index") * 100).round(1)
print("Counts:")
display(counts)
print("Row percentages (denominator = all bookings for that hotel):")
display(row_pct)

In [ ]:
# TODO: categorical x numerical with groupby, e.g.
# clean.groupby("cancel_status")["lead_time"].agg(["count", "mean", "median"])

In [ ]:
# TODO: time-based or numerical x numerical, e.g. monthly booking counts by hotel:
# monthly = clean.groupby([clean["arrival_date"].dt.to_period("M"), "hotel"]).size().unstack()

## 8. Final visualizations

At least **four** finished charts: a categorical comparison, a numerical distribution,
a relationship / grouped comparison, and a time-based or multi-group figure.
Each one needs a title, axis labels, a legend if needed, n, and a **What can we observe?** paragraph.

In [ ]:
# Figure 1 (example): cancellation rate by hotel
rate = clean.groupby("hotel")["is_canceled"].agg(["mean", "count"])
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(rate.index, rate["mean"] * 100, color=["#4C72B0", "#DD8452"])
ax.bar_label(bars, labels=[f"{v:.1f}%" for v in rate["mean"] * 100])
ax.set_title(f"Cancellation rate by hotel type (n = {rate['count'].sum()} bookings)")
ax.set_xlabel("Hotel type")
ax.set_ylabel("Cancelled bookings (%)")
ax.set_ylim(0, 100)
plt.tight_layout()
plt.show()

**What can we observe?** TODO: describe the main pattern with numbers. Don't claim a cause
(e.g. say "City Hotel bookings were cancelled more often", not "the city location causes cancellations").

In [ ]:
# TODO: Figure 2 – numerical distribution

**What can we observe?** TODO

In [ ]:
# TODO: Figure 3 – relationship or grouped comparison (e.g. box plot of lead time by cancellation status)

**What can we observe?** TODO

In [ ]:
# TODO: Figure 4 – time-based or multi-group figure (e.g. monthly bookings, one line per hotel)

**What can we observe?** TODO

## 9. Exports and verification

In [ ]:
# TODO: replace this example with the summary table your group finds most useful
summary = (clean.groupby("hotel")
           .agg(bookings=("is_canceled", "size"),
                cancellation_rate_pct=("is_canceled", lambda s: round(s.mean() * 100, 1)),
                mean_lead_time=("lead_time", "mean"),
                mean_adr=("average_daily_rate", "mean"),
                mean_total_nights=("total_nights", "mean"))
           .round(2)
           .reset_index())

clean.to_csv(CLEAN_FILE, index=False)
summary.to_csv(SUMMARY_FILE, index=False)

# Read both files back and verify them
check_clean = pd.read_csv(CLEAN_FILE, parse_dates=["arrival_date"])
check_summary = pd.read_csv(SUMMARY_FILE)
print("Cleaned file shape:", check_clean.shape, "| matches:", check_clean.shape == clean.shape)
print("Derived columns present:",
      all(c in check_clean.columns for c in ["arrival_date", "total_nights", "total_guests", "estimated_booking_value"]))
print("Summary file shape:", check_summary.shape)
check_summary

## 10. Conclusions, limitations and recommendation

**Answers to the research questions**
1. TODO
2. TODO
3. TODO

**Key findings (3–5, each backed by a number from the analysis)**
- TODO

**Limitations / cautions (at least 2)**
- TODO, e.g. a 1,000-booking sample from two hotels only, 2015–2017; associations are not causes.
- TODO

**Recommendation for hotel management:** TODO

**A next question these variables can't answer:** TODO, e.g. *why* guests cancel (there is no cancellation-reason variable).